# Nemotron LoRA — train on Kaggle (free 2xT4, 4-bit QLoRA)

**Run HEADLESS:** Save Version -> **Save & Run All (Commit)**.

**Add Input:** **competition data** + model **`nemotron-3-nano-30b-a3b-bf16`** (publisher `metric`). **GPU T4 x2, Internet On.**

Uses Kaggle's native **torch 2.10** as-is (no reinstall -> no torchvision break) and the matching prebuilt mamba wheels.

## 1. Code + dependencies (keep Kaggle's torch 2.10)

In [ ]:
%cd /kaggle/working
!rm -rf repo && git clone -b build/nemotron-pipeline https://github.com/SebAustin/NVIDIA-Nemotron-Model-Reasoning-Challenge repo
%cd repo
!pip install -q "transformers>=4.45,<5" peft trl datasets accelerate bitsandbytes psutil einops
# we never use vision; a mismatched torchvision breaks transformers' import -> remove it
!pip uninstall -y -q torchvision torchaudio
import torch
print("TORCH:", torch.__version__, "abi:", torch.compiled_with_cxx11_abi())

## 2. Install mamba_ssm only (torch_forward path; skips the T4-incompatible SSD kernel)

In [ ]:
# Install ONLY mamba_ssm (needed at import for rmsnorm), NOT causal_conv1d.
# Without causal_conv1d the model uses its torch_forward path, which skips the
# Mamba-2 SSD Triton kernel that fails to compile on T4 (sm_75).
!pip install -q --no-deps 'https://github.com/state-spaces/mamba/releases/download/v2.3.2.post1/mamba_ssm-2.3.2.post1+cu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl'
!python -c "import mamba_ssm; print('mamba_ssm OK; causal_conv1d intentionally absent -> torch_forward')"

## 3. Competition data (recursive find)

In [ ]:
import glob, os, shutil
os.makedirs('data', exist_ok=True)
hits = glob.glob('/kaggle/input/**/train.csv', recursive=True)
assert hits, "train.csv not found — Add Input -> the competition"
shutil.copy(hits[0], 'data/train.csv'); print('train.csv <-', hits[0])

## 4. EDA + build the SFT data

In [ ]:
!python scripts/01_eda.py --data-dir data
!python scripts/02_prepare_data.py --data-dir data

## 5. Train (4-bit QLoRA on 2xT4)
Base from the attached model mount (no 60 GB download). 1 epoch + seq 768 for a first finish; bump later. Smoke test runs first.

In [ ]:
import os
os.environ['QUANT'] = '4bit'
os.environ['NEMOTRON_MAX_MEMORY_GPU'] = '14GiB'
os.environ['SFT_MAX_SEQ_LENGTH'] = '768'
os.environ['NUM_EPOCHS'] = '1'
!python scripts/03_train_lora.py --data-path data/train_sft.jsonl --output-dir /kaggle/working/lora_adapter

## 6. Package the submission

In [ ]:
!python scripts/05_package_submission.py --adapter-dir /kaggle/working/lora_adapter --output /kaggle/working/submission.zip